In [ ]:
import os
import re
import csv
import json
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

In [ ]:
WINDOW_SIZE = "30min"
MIN_SEQ_LEN = 5
MAX_CHUNK_LEN = 64
STRIDE = 32

VAL_C_SIZE = 0.20
MAX_VAL_C_WINDOWS = 300

SERVER_NAME = "server_C_22"

SERVER_C_START_DATE = None
SERVER_C_END_DATE = None

FORCE_RECREATE_LOCKED_SPLIT = True

SAVE_DIR = "/content/drive/MyDrive/windows_logs_project"
VERSION = "serverC_22_30min_val_test"

In [ ]:
drive.mount("/content/drive")

os.makedirs(SAVE_DIR, exist_ok=True)

LOCKED_DATA_PATH = f"{SAVE_DIR}/serverC_22_locked_val_test_30min.pkl"
EXPORT_DIR = f"{SAVE_DIR}/serverC_22_export_30min"
PROMPT_EXPORT_DIR = f"{SAVE_DIR}/serverC_22_annotation_prompts_30min"

os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(PROMPT_EXPORT_DIR, exist_ok=True)

print("SAVE_DIR:", SAVE_DIR)
print("LOCKED_DATA_PATH:", LOCKED_DATA_PATH)
print("EXPORT_DIR:", EXPORT_DIR)
print("PROMPT_EXPORT_DIR:", PROMPT_EXPORT_DIR)

Mounted at /content/drive
SAVE_DIR: /content/drive/MyDrive/windows_logs_project
LOCKED_DATA_PATH: /content/drive/MyDrive/windows_logs_project/serverC_22_locked_val_test_30min.pkl
EXPORT_DIR: /content/drive/MyDrive/windows_logs_project/serverC_22_export_30min
PROMPT_EXPORT_DIR: /content/drive/MyDrive/windows_logs_project/serverC_22_annotation_prompts_30min


In [ ]:
WINDOWS_COLUMNS = [
    "Level",
    "Date and Time",
    "Source",
    "Event ID",
    "Task Category",
    "Message"
]


def load_windows_event_csv(file_path: str, log_name: str) -> pd.DataFrame:
    rows = []

    with open(file_path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
        reader = csv.reader(f)
        broken_header = next(reader, None)

        print(f"{log_name} | {Path(file_path).name} исходный header:", broken_header)

        for row in reader:
            if not row or all(not str(x).strip() for x in row):
                continue

            if len(row) == 6:
                rows.append(row)
            elif len(row) > 6:
                rows.append(row[:5] + [",".join(row[5:])])
            else:
                rows.append(row + [None] * (6 - len(row)))

    df = pd.DataFrame(rows, columns=WINDOWS_COLUMNS)
    df["LogName"] = log_name
    df["SourceFile"] = Path(file_path).name

    return df


def clean_windows_events(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ["Level", "Date and Time", "Source", "Event ID", "Task Category"]:
        df[col] = df[col].astype(str).str.strip()

    df["Message"] = df["Message"].fillna("").astype(str).str.strip()

    df["timestamp"] = pd.to_datetime(
        df["Date and Time"],
        errors="coerce",
        dayfirst=True
    )

    before = len(df)
    df = df.dropna(subset=["timestamp"]).copy()

    print(f"Удалено строк с некорректной датой: {before - len(df)}")

    return df


def load_many_windows_logs(files_by_logname: dict) -> pd.DataFrame:
    all_parts = []

    for log_name, paths in files_by_logname.items():
        for path in paths:
            raw = load_windows_event_csv(path, log_name)
            clean = clean_windows_events(raw)
            all_parts.append(clean)

    all_logs = pd.concat(all_parts, ignore_index=True)

    print("До удаления дубликатов:", len(all_logs))

    dedup_cols = [
        "LogName",
        "timestamp",
        "Level",
        "Source",
        "Event ID",
        "Task Category",
        "Message"
    ]

    duplicate_count = all_logs.duplicated(subset=dedup_cols).sum()
    print("Количество дубликатов:", duplicate_count)

    all_logs = all_logs.drop_duplicates(subset=dedup_cols, keep="first").copy()

    print("После удаления дубликатов:", len(all_logs))

    all_logs = all_logs.sort_values("timestamp").reset_index(drop=True)

    print("Диапазон дат:", all_logs["timestamp"].min(), "—", all_logs["timestamp"].max())

    return all_logs

In [ ]:
files_by_logname = {
    "System": [
        "/content/22 sys.csv"
    ],
    "Application": [
        "/content/22 app.csv"
    ]
}

windows_clean = load_many_windows_logs(files_by_logname)

windows_clean["Server"] = SERVER_NAME

print("Server:", SERVER_NAME)
print("До фильтрации по дате:", windows_clean.shape)
print("Диапазон дат:", windows_clean["timestamp"].min(), "—", windows_clean["timestamp"].max())

if SERVER_C_START_DATE is not None:
    windows_clean = windows_clean[
        windows_clean["timestamp"] >= pd.Timestamp(SERVER_C_START_DATE)
    ].copy()

if SERVER_C_END_DATE is not None:
    windows_clean = windows_clean[
        windows_clean["timestamp"] <= pd.Timestamp(SERVER_C_END_DATE)
    ].copy()

windows_clean = windows_clean.sort_values("timestamp").reset_index(drop=True)

print("После фильтрации по дате:", windows_clean.shape)
print("Диапазон дат:", windows_clean["timestamp"].min(), "—", windows_clean["timestamp"].max())

print("\nРаспределение Level:")
print(windows_clean["Level"].value_counts(dropna=False))
print(windows_clean["Level"].value_counts(normalize=True, dropna=False))

print("\nTop Event ID:")
display(
    windows_clean
    .groupby(["LogName", "Source", "Event ID", "Level"])
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name="count")
)

System | 22 sys.csv исходный header: ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category']
Удалено строк с некорректной датой: 0
Application | 22 app.csv исходный header: ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category']
Удалено строк с некорректной датой: 3
До удаления дубликатов: 104746
Количество дубликатов: 25889
После удаления дубликатов: 78857
Диапазон дат: 2026-04-07 18:03:02 — 2026-05-05 07:50:19
Server: server_C_22
До фильтрации по дате: (78857, 10)
Диапазон дат: 2026-04-07 18:03:02 — 2026-05-05 07:50:19
После фильтрации по дате: (78857, 10)
Диапазон дат: 2026-04-07 18:03:02 — 2026-05-05 07:50:19

Распределение Level:
Level
Information    77818
Error            606
Warning          433
Name: count, dtype: int64
Level
Information    0.986824
Error          0.007685
Warning        0.005491
Name: proportion, dtype: float64

Top Event ID:


,LogName,Source,Event ID,Level,count
0,System,Service Control Manager,7036,Information,63370
1,Application,Windows Error Reporting,1001,Information,7145
2,Application,MSSQLSERVER,18265,Information,1689
3,Application,MSSQLSERVER,3014,Information,1120
4,Application,MSSQLSERVER,18270,Information,914
5,System,Service Control Manager,7040,Information,768
6,Application,MSSQLSERVER,14161,Information,573
7,Application,MSSQLSERVER,14160,Information,573
8,Application,MSSQLSERVER,18053,Error,509
9,System,Microsoft-Windows-GroupPolicy,1502,Information,379


## Нормализация сообщений

In [ ]:
def normalize_message(text: str) -> str:
    text = str(text).lower()


    text = re.sub(
        r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b",
        "<guid>",
        text
    )


    text = re.sub(
        r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
        "<ip>",
        text
    )


    text = re.sub(
        r"[a-z]:\\(?:[^\\/:*?\"<>|\r\n]+\\)*[^\\/:*?\"<>|\r\n]*",
        "<path>",
        text
    )


    text = re.sub(
        r"\b\d{1,2}[./-]\d{1,2}[./-]\d{2,4}\b",
        "<date>",
        text
    )


    text = re.sub(
        r"\b\d{1,2}:\d{2}(:\d{2})?\b",
        "<time>",
        text
    )


    text = re.sub(
        r"\b0x[0-9a-f]+\b",
        "<hex>",
        text
    )


    text = re.sub(
        r"\b\d+\b",
        "<num>",
        text
    )


    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
windows_clean["NormalizedMessage"] = windows_clean["Message"].apply(normalize_message)

display(windows_clean[[
    "timestamp",
    "LogName",
    "Level",
    "Source",
    "Event ID",
    "Task Category",
    "Message",
    "NormalizedMessage"
]].head(10))

,timestamp,LogName,Level,Source,Event ID,Task Category,Message,NormalizedMessage
0,2026-04-07 18:03:02,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
1,2026-04-07 18:04:59,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
2,2026-04-07 18:07:55,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
3,2026-04-07 18:08:04,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
4,2026-04-07 18:08:48,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
5,2026-04-07 18:09:06,System,Information,Service Control Manager,7036,None,The WMI Performance Adapter service entered th...,the wmi performance adapter service entered th...
6,2026-04-07 18:10:17,System,Information,Service Control Manager,7036,None,The WMI Performance Adapter service entered th...,the wmi performance adapter service entered th...
7,2026-04-07 18:10:29,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
8,2026-04-07 18:11:16,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
9,2026-04-07 18:11:26,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...


## Словарь событий

In [ ]:
windows_clean["EventToken"] = windows_clean.apply(
    lambda row: (
        f"{row['LogName']} | "
        f"{row['Source']} | "
        f"{row['Event ID']} | "
        f"{row['Level']} | "
        f"{row['NormalizedMessage']}"
    ),
    axis=1
)

windows_clean["EventTemplateForReview"] = windows_clean["EventToken"]

print("Уникальных EventToken:", windows_clean["EventToken"].nunique())

display(windows_clean[[
    "timestamp",
    "LogName",
    "Level",
    "Source",
    "Event ID",
    "EventToken"
]].head(10))

Уникальных EventToken: 2989


,timestamp,LogName,Level,Source,Event ID,EventToken
0,2026-04-07 18:03:02,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
1,2026-04-07 18:04:59,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
2,2026-04-07 18:07:55,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
3,2026-04-07 18:08:04,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
4,2026-04-07 18:08:48,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
5,2026-04-07 18:09:06,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
6,2026-04-07 18:10:17,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
7,2026-04-07 18:10:29,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
8,2026-04-07 18:11:16,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
9,2026-04-07 18:11:26,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...


In [ ]:
unique_tokens = (
    windows_clean["EventToken"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

template_to_id = {
    token: f"W{i + 1}"
    for i, token in enumerate(unique_tokens)
}

id_to_template = {
    v: k
    for k, v in template_to_id.items()
}

windows_clean["EventTemplateID"] = windows_clean["EventToken"].map(template_to_id)

print("Количество шаблонов Windows:", len(template_to_id))

display(windows_clean[[
    "timestamp",
    "LogName",
    "Level",
    "Source",
    "Event ID",
    "EventTemplateID",
    "EventToken"
]].head(10))

Количество шаблонов Windows: 2989


,timestamp,LogName,Level,Source,Event ID,EventTemplateID,EventToken
0,2026-04-07 18:03:02,System,Information,Service Control Manager,7036,W2959,System | Service Control Manager | 7036 | Info...
1,2026-04-07 18:04:59,System,Information,Service Control Manager,7036,W2960,System | Service Control Manager | 7036 | Info...
2,2026-04-07 18:07:55,System,Information,Service Control Manager,7036,W2959,System | Service Control Manager | 7036 | Info...
3,2026-04-07 18:08:04,System,Information,Service Control Manager,7036,W2960,System | Service Control Manager | 7036 | Info...
4,2026-04-07 18:08:48,System,Information,Service Control Manager,7036,W2959,System | Service Control Manager | 7036 | Info...
5,2026-04-07 18:09:06,System,Information,Service Control Manager,7036,W2984,System | Service Control Manager | 7036 | Info...
6,2026-04-07 18:10:17,System,Information,Service Control Manager,7036,W2983,System | Service Control Manager | 7036 | Info...
7,2026-04-07 18:10:29,System,Information,Service Control Manager,7036,W2960,System | Service Control Manager | 7036 | Info...
8,2026-04-07 18:11:16,System,Information,Service Control Manager,7036,W2959,System | Service Control Manager | 7036 | Info...
9,2026-04-07 18:11:26,System,Information,Service Control Manager,7036,W2960,System | Service Control Manager | 7036 | Info...


## Исключаем явные аномалии

In [ ]:
HARD_CRITICAL_EVENT_IDS = {
    "41",
    "6008",
    "7031",
    "7034",
    "17832",
    "17883",
    "1102",
    "4625",
    "4740",
    "9002",
    "4014",
    "18456"
}

SOFT_SUSPICIOUS_EVENT_IDS = {
    "1000",
    "1001",
    "10010",
    "10036",
    "1801",
    "1202",
    "1065",
    "8193"

## Временные окна

In [ ]:
def build_sequences_by_time_window(
    df: pd.DataFrame,
    window: str = "30min",
    min_seq_len: int = 5
) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values("timestamp").reset_index(drop=True)

    df["window_start"] = df["timestamp"].dt.floor(window)

    sequences = []

    for window_start, group in df.groupby("window_start"):
        group = group.sort_values("timestamp")

        features = group["EventTemplateID"].tolist()
        timestamps = group["timestamp"].tolist()

        if len(features) < min_seq_len:
            continue

        time_intervals = [0.0]

        for i in range(1, len(timestamps)):
            delta = (timestamps[i] - timestamps[i - 1]).total_seconds()
            time_intervals.append(float(delta))

        latency = float((timestamps[-1] - timestamps[0]).total_seconds())

        sequences.append({
            "WindowSize": window,
            "WindowStart": window_start,
            "WindowEnd": window_start + pd.Timedelta(window),

            "Features": features,
            "TimeInterval": time_intervals,
            "Latency": latency,
            "SeqLen": len(features),


            "LogNames": sorted(group["LogName"].astype(str).unique().tolist()),
            "Sources": sorted(group["Source"].astype(str).unique().tolist()),
            "Levels": sorted(group["Level"].astype(str).unique().tolist()),
            "EventIDs": sorted(group["Event ID"].astype(str).unique().tolist()),

            "EventLogNames": group["LogName"].astype(str).tolist(),
            "EventSources": group["Source"].astype(str).tolist(),
            "EventLevels": group["Level"].astype(str).tolist(),
            "EventIDsPerEvent": group["Event ID"].astype(str).tolist(),
            "EventTokens": group["EventToken"].astype(str).tolist(),

            "RawMessages": group["Message"].astype(str).tolist(),
            "ReviewTemplates": group["EventTemplateForReview"].astype(str).tolist(),

            "Label": None
        })

    return pd.DataFrame(sequences)

## Отбор нормальных окон для train

In [ ]:
def recompute_weak_label_for_row(row, q95_len=None):
    levels = [str(x).lower() for x in row["Levels"]]
    event_ids = set(str(x) for x in row["EventIDs"])
    sources = [str(x).lower() for x in row["Sources"]]

    hard_reasons = []
    soft_reasons = []

    # hard reasons
    if any("critical" in x for x in levels):
        hard_reasons.append("critical_level")

    matched_hard_ids = event_ids.intersection(HARD_CRITICAL_EVENT_IDS)
    if matched_hard_ids:
        hard_reasons.append(
            "hard_critical_event_id:" + ",".join(sorted(matched_hard_ids))
        )

    if any("mssqlserver" in s for s in sources) and any("error" in x for x in levels):
        hard_reasons.append("mssql_error")

    if any("error" in x for x in levels):
        soft_reasons.append("error_level")

    matched_soft_ids = event_ids.intersection(SOFT_SUSPICIOUS_EVENT_IDS)
    if matched_soft_ids:
        soft_reasons.append(
            "soft_suspicious_event_id:" + ",".join(sorted(matched_soft_ids))
        )

    if q95_len is not None and row["SeqLen"] > q95_len:
        soft_reasons.append("high_event_volume")

    weak_label = int(len(hard_reasons) > 0 or len(soft_reasons) > 0)
    hard_weak_label = int(len(hard_reasons) > 0)

    row["WeakLabel"] = weak_label
    row["HardWeakLabel"] = hard_weak_label
    row["HardSuspiciousReason"] = "; ".join(hard_reasons)
    row["SoftSuspiciousReason"] = "; ".join(soft_reasons)
    row["SuspiciousReason"] = "; ".join(hard_reasons + soft_reasons)
    row["NeedsReview"] = weak_label == 1

    return row


def add_weak_labels(windows_seq: pd.DataFrame) -> pd.DataFrame:
    windows_seq = windows_seq.copy()

    q95_len = windows_seq["SeqLen"].quantile(0.95)

    rows = []

    for _, row in windows_seq.iterrows():
        rows.append(recompute_weak_label_for_row(row.copy(), q95_len=q95_len))

    return pd.DataFrame(rows).reset_index(drop=True)

## Сравнение окон

In [ ]:
windows_seq_15 = build_sequences_by_time_window(
    windows_clean,
    window="15min",
    min_seq_len=MIN_SEQ_LEN
)

windows_seq_30 = build_sequences_by_time_window(
    windows_clean,
    window="30min",
    min_seq_len=MIN_SEQ_LEN
)

windows_seq_15 = add_weak_labels(windows_seq_15)
windows_seq_30 = add_weak_labels(windows_seq_30)

print("15 минут:", windows_seq_15.shape)
print("30 минут:", windows_seq_30.shape)

15 минут: (2648, 25)
30 минут: (1324, 25)


In [ ]:
def summarize_windows_sequences(name, seq_df):
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Количество окон:", len(seq_df))
    print("Диапазон времени:", seq_df["WindowStart"].min(), "—", seq_df["WindowEnd"].max())

    print("\nДлины последовательностей:")
    display(seq_df["SeqLen"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

    print("\nWeakLabel distribution:")
    print(seq_df["WeakLabel"].value_counts(dropna=False))
    print(seq_df["WeakLabel"].value_counts(normalize=True, dropna=False))

    print("\nТоп-10 самых длинных окон:")
    display(
        seq_df.sort_values("SeqLen", ascending=False).head(10)[[
            "WindowStart",
            "WindowEnd",
            "SeqLen",
            "Levels",
            "EventIDs",
            "WeakLabel",
            "SuspiciousReason"
        ]]
    )


summarize_windows_sequences("Окна 15 минут", windows_seq_15)
summarize_windows_sequences("Окна 30 минут", windows_seq_30)


Окна 15 минут
Количество окон: 2648
Диапазон времени: 2026-04-07 18:00:00 — 2026-05-05 08:00:00

Длины последовательностей:


,SeqLen
count,2648.000000
mean,29.779834
std,18.318207
min,14.000000
50%,25.000000
75%,30.000000
90%,38.000000
95%,73.000000
99%,93.060000
max,352.000000



WeakLabel distribution:
WeakLabel
0    1798
1     850
Name: count, dtype: int64
WeakLabel
0    0.679003
1    0.320997
Name: proportion, dtype: float64

Топ-10 самых длинных окон:


,WindowStart,WindowEnd,SeqLen,Levels,EventIDs,WeakLabel,SuspiciousReason
1915,2026-04-27 16:45:00,2026-04-27 17:00:00,352,[Information],"[14160, 14161, 7036]",1,high_event_volume
1914,2026-04-27 16:30:00,2026-04-27 16:45:00,334,"[Information, Warning]","[1202, 14160, 14161, 1502, 158, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
1913,2026-04-27 16:15:00,2026-04-27 16:30:00,248,"[Error, Information]","[14160, 14161, 18053, 7036]",1,mssql_error; error_level; high_event_volume
1912,2026-04-27 16:00:00,2026-04-27 16:15:00,181,"[Error, Information]","[1008, 14160, 14161, 18265, 7036]",1,mssql_error; error_level; high_event_volume
1468,2026-04-23 01:00:00,2026-04-23 01:15:00,163,[Information],"[1001, 18264, 18265, 18270, 3014, 7036]",1,soft_suspicious_event_id:1001; high_event_volume
1916,2026-04-27 17:00:00,2026-04-27 17:15:00,150,[Information],"[14160, 14161, 18265, 7036]",1,high_event_volume
500,2026-04-12 23:00:00,2026-04-12 23:15:00,137,[Information],"[17137, 18265, 49930, 7036]",1,high_event_volume
700,2026-04-15 01:00:00,2026-04-15 01:15:00,120,"[Information, Warning]","[1202, 1502, 158, 18264, 18265, 18270, 3014, 7...",1,soft_suspicious_event_id:1202; high_event_volume
2044,2026-04-29 01:00:00,2026-04-29 01:15:00,116,[Information],"[18264, 18265, 18270, 3014, 7036]",1,high_event_volume
2620,2026-05-05 01:00:00,2026-05-05 01:15:00,113,[Information],"[18264, 18265, 18270, 3014, 7036]",1,high_event_volume



Окна 30 минут
Количество окон: 1324
Диапазон времени: 2026-04-07 18:00:00 — 2026-05-05 08:00:00

Длины последовательностей:


,SeqLen
count,1324.000000
mean,59.559668
std,29.324961
min,33.000000
50%,52.000000
75%,61.000000
90%,97.000000
95%,109.000000
99%,135.770000
max,686.000000



WeakLabel distribution:
WeakLabel
1    688
0    636
Name: count, dtype: int64
WeakLabel
1    0.519637
0    0.480363
Name: proportion, dtype: float64

Топ-10 самых длинных окон:


,WindowStart,WindowEnd,SeqLen,Levels,EventIDs,WeakLabel,SuspiciousReason
957,2026-04-27 16:30:00,2026-04-27 17:00:00,686,"[Information, Warning]","[1202, 14160, 14161, 1502, 158, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
956,2026-04-27 16:00:00,2026-04-27 16:30:00,429,"[Error, Information]","[1008, 14160, 14161, 18053, 18265, 7036]",1,mssql_error; error_level; high_event_volume
734,2026-04-23 01:00:00,2026-04-23 01:30:00,186,[Information],"[1001, 18264, 18265, 18270, 3014, 7036]",1,soft_suspicious_event_id:1001; high_event_volume
1157,2026-05-01 20:30:00,2026-05-01 21:00:00,178,[Information],"[1001, 17137, 49930, 7036]",1,soft_suspicious_event_id:1001; high_event_volume
958,2026-04-27 17:00:00,2026-04-27 17:30:00,177,[Information],"[14160, 14161, 18265, 7036]",1,high_event_volume
250,2026-04-12 23:00:00,2026-04-12 23:30:00,157,[Information],"[17137, 18265, 49930, 7036]",1,high_event_volume
1310,2026-05-05 01:00:00,2026-05-05 01:30:00,146,"[Information, Warning]","[1202, 1502, 158, 18264, 18265, 18270, 3014, 7...",1,soft_suspicious_event_id:1202; high_event_volume
350,2026-04-15 01:00:00,2026-04-15 01:30:00,144,"[Information, Warning]","[1202, 1502, 158, 18264, 18265, 18270, 3014, 7...",1,soft_suspicious_event_id:1202; high_event_volume
1022,2026-04-29 01:00:00,2026-04-29 01:30:00,139,[Information],"[18264, 18265, 18270, 3014, 7036]",1,high_event_volume
830,2026-04-25 01:00:00,2026-04-25 01:30:00,139,[Information],"[18264, 18265, 18270, 3014, 7036, 7040]",1,high_event_volume


In [ ]:
comparison_rows = []

for name, seq_df in [
    ("15min", windows_seq_15),
    ("30min", windows_seq_30)
]:
    comparison_rows.append({
        "window": name,
        "num_windows": len(seq_df),
        "mean_len": seq_df["SeqLen"].mean(),
        "median_len": seq_df["SeqLen"].median(),
        "p90_len": seq_df["SeqLen"].quantile(0.90),
        "p95_len": seq_df["SeqLen"].quantile(0.95),
        "p99_len": seq_df["SeqLen"].quantile(0.99),
        "max_len": seq_df["SeqLen"].max(),
        "weak_anomaly_count": int(seq_df["WeakLabel"].sum()),
        "weak_anomaly_ratio": seq_df["WeakLabel"].mean()
    })

window_comparison = pd.DataFrame(comparison_rows)

display(window_comparison)

,window,num_windows,mean_len,median_len,p90_len,p95_len,p99_len,max_len,weak_anomaly_count,weak_anomaly_ratio
0,15min,2648,29.779834,25.0,38.0,73.0,93.06,352,850,0.320997
1,30min,1324,59.559668,52.0,97.0,109.0,135.77,686,688,0.519637


In [ ]:
windows_seq = windows_seq_30.copy()

print("Используем окно:", WINDOW_SIZE)
print("windows_seq:", windows_seq.shape)

Используем окно: 30min
windows_seq: (1324, 25)


In [ ]:
def has_level(row, level_name: str) -> bool:
    return any(level_name.lower() in str(x).lower() for x in row["Levels"])


def is_safe_normal(row, q95_len_train) -> bool:
    levels = [str(x).lower() for x in row["Levels"]]
    sources = [str(x).lower() for x in row["Sources"]]
    event_ids = set(str(x) for x in row["EventIDs"])

    has_critical_level = any("critical" in x for x in levels)

    has_hard_critical_event_id = (
        len(event_ids.intersection(HARD_CRITICAL_EVENT_IDS)) > 0
    )

    high_volume = row["SeqLen"] > q95_len_train

    has_error_level = any("error" in x for x in levels)

    serious_error_source = (
        any("mssqlserver" in s for s in sources)
        or any("service control manager" in s for s in sources)
        or any("kernel-power" in s for s in sources)
        or any("eventlog" in s for s in sources)
        or any("sqlserveragent" in s for s in sources)
    )

    has_serious_error = has_error_level and serious_error_source

    return (
        not has_critical_level
        and not has_hard_critical_event_id
        and not high_volume
        and not has_serious_error
    )

In [ ]:
def split_long_sequence_row(row, max_len=64, stride=32, q95_len_for_weak=None):
    n = len(row["Features"])

    per_event_cols = [
        "Features",
        "TimeInterval",
        "RawMessages",
        "ReviewTemplates",
        "EventLogNames",
        "EventSources",
        "EventLevels",
        "EventIDsPerEvent",
        "EventTokens"
    ]

    for col in per_event_cols:
        if col not in row:
            raise ValueError(f"В row отсутствует колонка {col}. Проверь build_sequences_by_time_window.")

    if n <= max_len:
        new_row = row.copy()

        new_row["ChunkID"] = 0
        new_row["OriginalSeqLen"] = n
        new_row["ChunkStartPos"] = 0
        new_row["ChunkEndPos"] = n
        new_row["WasChunked"] = False

        new_row["LogNames"] = sorted(set(map(str, new_row["EventLogNames"])))
        new_row["Sources"] = sorted(set(map(str, new_row["EventSources"])))
        new_row["Levels"] = sorted(set(map(str, new_row["EventLevels"])))
        new_row["EventIDs"] = sorted(set(map(str, new_row["EventIDsPerEvent"])))

        new_row = recompute_weak_label_for_row(
            new_row,
            q95_len=q95_len_for_weak
        )

        return [new_row]

    chunks = []

    starts = list(range(0, n - max_len + 1, stride))
    last_start = n - max_len

    if starts[-1] != last_start:
        starts.append(last_start)

    for chunk_id, start in enumerate(starts):
        end = start + max_len

        new_row = row.copy()

        for col in per_event_cols:
            new_row[col] = row[col][start:end]

        new_row["SeqLen"] = len(new_row["Features"])
        new_row["Latency"] = float(sum(new_row["TimeInterval"]))
        new_row["LogNames"] = sorted(set(map(str, new_row["EventLogNames"])))
        new_row["Sources"] = sorted(set(map(str, new_row["EventSources"])))
        new_row["Levels"] = sorted(set(map(str, new_row["EventLevels"])))
        new_row["EventIDs"] = sorted(set(map(str, new_row["EventIDsPerEvent"])))

        new_row["ChunkID"] = chunk_id
        new_row["OriginalSeqLen"] = n
        new_row["ChunkStartPos"] = start
        new_row["ChunkEndPos"] = end
        new_row["WasChunked"] = True

        new_row = recompute_weak_label_for_row(
            new_row,
            q95_len=q95_len_for_weak
        )

        chunks.append(new_row)

    return chunks


def split_long_sequences(df, max_len=64, stride=32, q95_len_for_weak=None):
    all_rows = []

    for _, row in df.iterrows():
        all_rows.extend(
            split_long_sequence_row(
                row,
                max_len=max_len,
                stride=stride,
                q95_len_for_weak=q95_len_for_weak
            )
        )

    return pd.DataFrame(all_rows).reset_index(drop=True)

## Split

In [ ]:
if FORCE_RECREATE_LOCKED_SPLIT and os.path.exists(LOCKED_DATA_PATH):
    os.remove(LOCKED_DATA_PATH)
    print("Удалён старый locked split:", LOCKED_DATA_PATH)


if os.path.exists(LOCKED_DATA_PATH):
    print("Найден locked Server C split. Загружаю существующее разбиение...")

    with open(LOCKED_DATA_PATH, "rb") as f:
        locked_data = pickle.load(f)

    val_C_raw = locked_data["val_C_raw"]
    test_C_raw = locked_data["test_C_raw"]

    val_C = locked_data["val_C"]
    test_C = locked_data["test_C"]

    annotation_df_C = locked_data["annotation_df_C"]

    template_to_id = locked_data["template_to_id"]
    id_to_template = locked_data["id_to_template"]

    print("Loaded locked Server C data:")
    print("val_C_raw:", val_C_raw.shape)
    print("test_C_raw:", test_C_raw.shape)
    print("val_C:", val_C.shape)
    print("test_C:", test_C.shape)
    print("annotation_df_C:", annotation_df_C.shape)

else:
    print("Locked Server C split не найден. Создаю val_C / test_C...")

    windows_seq_sorted = windows_seq.sort_values("WindowStart").reset_index(drop=True)

    n = len(windows_seq_sorted)

    val_end = min(
        int(n * VAL_C_SIZE),
        MAX_VAL_C_WINDOWS
    )

    val_C_raw = windows_seq_sorted.iloc[:val_end].copy()
    test_C_raw = windows_seq_sorted.iloc[val_end:].copy()

    val_C_raw["Split"] = "val_C"
    test_C_raw["Split"] = "test_C"

    print("Всего окон Server C:", n)
    print("val_C_raw:", val_C_raw.shape)
    print("test_C_raw:", test_C_raw.shape)

    print("\nДиапазоны времени:")
    print("val_C:", val_C_raw["WindowStart"].min(), "—", val_C_raw["WindowEnd"].max())
    print("test_C:", test_C_raw["WindowStart"].min(), "—", test_C_raw["WindowEnd"].max())

    if len(val_C_raw) > 0 and len(test_C_raw) > 0:
        assert val_C_raw["WindowEnd"].max() <= test_C_raw["WindowStart"].min()

    q95_len_C = windows_seq_sorted["SeqLen"].quantile(0.95)

    print("\n95-й перцентиль длины окон Server C:", q95_len_C)

    val_C = split_long_sequences(
        val_C_raw,
        max_len=MAX_CHUNK_LEN,
        stride=STRIDE,
        q95_len_for_weak=q95_len_C
    )

    test_C = split_long_sequences(
        test_C_raw,
        max_len=MAX_CHUNK_LEN,
        stride=STRIDE,
        q95_len_for_weak=q95_len_C
    )

    val_C["Split"] = "val_C"
    test_C["Split"] = "test_C"

    print("\nПосле chunking:")
    print("val_C_raw:", val_C_raw.shape, "->", val_C.shape)
    print("test_C_raw:", test_C_raw.shape, "->", test_C.shape)

    print("\nМаксимальные длины после chunking:")
    print("val_C:", val_C["SeqLen"].max())
    print("test_C:", test_C["SeqLen"].max())

    annotation_df_C = pd.concat([
        val_C,
        test_C
    ]).reset_index(drop=True)

    annotation_df_C["AnnotationID"] = range(len(annotation_df_C))

    annotation_df_C["StableID"] = (
        annotation_df_C["Split"].astype(str) + "_" +
        annotation_df_C["WindowStart"].astype(str) + "_" +
        annotation_df_C["WindowEnd"].astype(str) + "_" +
        annotation_df_C["ChunkID"].astype(str) + "_" +
        annotation_df_C["ChunkStartPos"].astype(str) + "_" +
        annotation_df_C["ChunkEndPos"].astype(str)
    )

    locked_data = {
        "params": {
            "VERSION": VERSION,
            "WINDOW_SIZE": WINDOW_SIZE,
            "MIN_SEQ_LEN": MIN_SEQ_LEN,
            "MAX_CHUNK_LEN": MAX_CHUNK_LEN,
            "STRIDE": STRIDE,
            "VAL_C_SIZE": VAL_C_SIZE,
            "MAX_VAL_C_WINDOWS": MAX_VAL_C_WINDOWS,
            "SERVER_NAME": SERVER_NAME,
            "SERVER_C_START_DATE": SERVER_C_START_DATE,
            "SERVER_C_END_DATE": SERVER_C_END_DATE,
            "q95_len_C": q95_len_C,
            "HARD_CRITICAL_EVENT_IDS": list(HARD_CRITICAL_EVENT_IDS),
            "SOFT_SUSPICIOUS_EVENT_IDS": list(SOFT_SUSPICIOUS_EVENT_IDS),
        },
        "val_C_raw": val_C_raw,
        "test_C_raw": test_C_raw,
        "val_C": val_C,
        "test_C": test_C,
        "annotation_df_C": annotation_df_C,
        "template_to_id": template_to_id,
        "id_to_template": id_to_template
    }

    with open(LOCKED_DATA_PATH, "wb") as f:
        pickle.dump(locked_data, f)

    print("\nLocked Server C split saved:", LOCKED_DATA_PATH)

Locked Server C split не найден. Создаю val_C / test_C...
Всего окон Server C: 1324
val_C_raw: (264, 26)
test_C_raw: (1060, 26)

Диапазоны времени:
val_C: 2026-04-07 18:00:00 — 2026-04-13 06:00:00
test_C: 2026-04-13 06:00:00 — 2026-05-05 08:00:00

95-й перцентиль длины окон Server C: 109.0

После chunking:
val_C_raw: (264, 26) -> (300, 31)
test_C_raw: (1060, 26) -> (1473, 31)

Максимальные длины после chunking:
val_C: 64
test_C: 64

Locked Server C split saved: /content/drive/MyDrive/windows_logs_project/serverC_22_locked_val_test_30min.pkl


In [ ]:
def show_server_c_split_sizes():
    print("=" * 70)
    print("SERVER C DATASET SIZES")
    print("=" * 70)

    print("\nRaw windows:")
    print("val_C_raw:  ", val_C_raw.shape)
    print("test_C_raw: ", test_C_raw.shape)

    print("\nAfter chunking:")
    print("val_C:      ", val_C.shape)
    print("test_C:     ", test_C.shape)

    print("\nAnnotation:")
    print("annotation_df_C:", annotation_df_C.shape)

    print("\nTime ranges:")
    print("val_C: ", val_C["WindowStart"].min(), "—", val_C["WindowEnd"].max())
    print("test_C:", test_C["WindowStart"].min(), "—", test_C["WindowEnd"].max())

    print("\nSequence length stats:")
    length_stats = pd.DataFrame({
        "val_C": val_C["SeqLen"].describe(),
        "test_C": test_C["SeqLen"].describe()
    })

    display(length_stats)

    print("\nWeakLabel distribution:")
    for name, df in [
        ("val_C", val_C),
        ("test_C", test_C),
        ("annotation_df_C", annotation_df_C)
    ]:
        if "WeakLabel" in df.columns:
            print(f"\n{name}:")
            print(df["WeakLabel"].value_counts(dropna=False))
            print(df["WeakLabel"].value_counts(normalize=True, dropna=False))


show_server_c_split_sizes()

SERVER C DATASET SIZES

Raw windows:
val_C_raw:   (264, 26)
test_C_raw:  (1060, 26)

After chunking:
val_C:       (300, 31)
test_C:      (1473, 31)

Annotation:
annotation_df_C: (1773, 33)

Time ranges:
val_C:  2026-04-07 18:00:00 — 2026-04-13 06:00:00
test_C: 2026-04-13 06:00:00 — 2026-05-05 08:00:00

Sequence length stats:


,val_C,test_C
count,300.000000,1473.000000
mean,52.640000,56.309572
std,8.427761,8.462306
min,33.000000,34.000000
25%,46.000000,49.000000
50%,51.000000,60.000000
75%,62.000000,64.000000
max,64.000000,64.000000



WeakLabel distribution:

val_C:
WeakLabel
0    229
1     71
Name: count, dtype: int64
WeakLabel
0    0.763333
1    0.236667
Name: proportion, dtype: float64

test_C:
WeakLabel
1    939
0    534
Name: count, dtype: int64
WeakLabel
1    0.637475
0    0.362525
Name: proportion, dtype: float64

annotation_df_C:
WeakLabel
1    1010
0     763
Name: count, dtype: int64
WeakLabel
1    0.569656
0    0.430344
Name: proportion, dtype: float64


In [ ]:
print("Уникальных последовательностей:")
print("val_seq:     ", val_C["Features"].apply(tuple).nunique())
print("test_seq:    ", test_C["Features"].apply(tuple).nunique())

print("\nВсего строк:")
print("val_seq:     ", len(val_C))
print("test_seq:    ", len(test_C))

Уникальных последовательностей:
val_seq:      300
test_seq:     1473

Всего строк:
val_seq:      300
test_seq:     1473


## Разметка val/test

In [ ]:
def shorten_text(text, max_chars=500):
    text = str(text).replace("\n", " ").replace("\r", " ")
    text = " ".join(text.split())

    if len(text) <= max_chars:
        return text

    return text[:max_chars] + " ...[truncated]"

In [ ]:
def build_annotation_prompt(row, max_events=80, max_message_chars=500):
    annotation_id = row["AnnotationID"]
    stable_id = row["StableID"]
    split = row["Split"]

    window_start = row["WindowStart"]
    window_end = row["WindowEnd"]
    seq_len = row["SeqLen"]

    levels = row.get("Levels", [])
    sources = row.get("Sources", [])
    event_ids = row.get("EventIDs", [])
    suspicious_reason = row.get("SuspiciousReason", "")

    features = row.get("Features", [])
    time_intervals = row.get("TimeInterval", [])
    review_templates = row.get("ReviewTemplates", [])
    raw_messages = row.get("RawMessages", [])

    event_lines = []

    n_events = min(len(features), max_events)

    for i in range(n_events):
        feature = features[i] if i < len(features) else ""
        delta_t = time_intervals[i] if i < len(time_intervals) else ""
        template = review_templates[i] if i < len(review_templates) else ""
        message = raw_messages[i] if i < len(raw_messages) else ""

        event_lines.append(
            f"{i + 1}. "
            f"Token: {feature} | "
            f"DeltaTimeSec: {delta_t} | "
            f"Template: {shorten_text(template, max_message_chars)} | "
            f"Message: {shorten_text(message, max_message_chars)}"
        )

    if len(features) > max_events:
        event_lines.append(
            f"... Остальные события не показаны: {len(features) - max_events}"
        )

    events_text = "\n".join(event_lines)

    prompt = f"""
Ты эксперт по анализу Windows Event Logs и обнаружению аномалий в последовательностях событий.

Нужно разметить одну последовательность событий из журналов Windows System/Application.

Классы:

0 = normal
Последовательность похожа на обычную активность системы: информационные события, обычный запуск/остановка служб, Windows Update, BITS, Group Policy, WMI/SCM без признаков сбоя.

1 = anomaly
Последовательность содержит признаки сбоя или нештатного поведения: Error/Critical, Windows Error Reporting, ошибки SQL Server, failed login, unexpected shutdown/restart, service terminated unexpectedly, audit log cleared, repeated crash reports, массовая остановка критичных служб или другая явно подозрительная цепочка.

Важно:
- Не считай Service Control Manager 7036 сам по себе аномалией.
- Не считай обычные циклы WMI Performance Adapter running/stopped аномалией без дополнительных признаков сбоя.
- Не считай Windows Update, BITS, Group Policy или AppX активность аномалией, если нет Error/Critical/WER/SQL-сбоев.
- DCOM timeout или Warning может быть аномалией только в контексте повторяемости, связи с ошибками или другими сбойными событиями.
- Если признаки слабые и нет явных ошибок, выбери label=0, но можешь поставить повышенный anomaly_score.
- Если есть повторяющиеся WER 1001 по sqlservr.exe, MSSQLSERVER Error, failed job SQL Agent, unexpected shutdown/restart или hard critical Event ID, выбери label=1.

Верни ответ строго в JSON формате без дополнительного текста:

{{
  "label": 0 или 1,
  "anomaly_score": число от 0.0 до 1.0,
  "confidence": число от 0.0 до 1.0,
  "key_events": ["список наиболее важных Event ID / источников / токенов"],
  "reason": "краткое объяснение на русском языке"
}}

Метаданные последовательности:
AnnotationID: {annotation_id}
StableID: {stable_id}
Split: {split}
WindowStart: {window_start}
WindowEnd: {window_end}
SeqLen: {seq_len}

Levels в chunk-е:
{levels}

Sources в chunk-е:
{sources}

EventIDs в chunk-е:
{event_ids}

Предварительные weak-label причины chunk-а:
{suspicious_reason}

События последовательности:
{events_text}
""".strip()

    return prompt

In [ ]:
annotation_prompts = annotation_df_C.copy()

annotation_prompts["Prompt"] = annotation_prompts.apply(
    lambda row: build_annotation_prompt(
        row,
        max_events=80,
        max_message_chars=500
    ),
    axis=1
)

prompt_columns = [
    "AnnotationID",
    "StableID",
    "Split",
    "WindowStart",
    "WindowEnd",
    "SeqLen",
    "Levels",
    "Sources",
    "EventIDs",
    "SuspiciousReason",
    "Prompt"
]

annotation_prompts_export = annotation_prompts[prompt_columns].copy()

list_cols = [
    "Levels",
    "Sources",
    "EventIDs"
]

for col in list_cols:
    annotation_prompts_export[col] = annotation_prompts_export[col].apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
    )
ANNOTATION_PROMPTS_CSV_PATH = f"{PROMPT_EXPORT_DIR}/serverC_22_annotation_prompts_30min.csv"
ANNOTATION_PROMPTS_JSONL_PATH = f"{PROMPT_EXPORT_DIR}/serverC_22_annotation_prompts_30min.jsonl"

annotation_prompts_export.to_csv(
    ANNOTATION_PROMPTS_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("CSV saved:", ANNOTATION_PROMPTS_CSV_PATH)
print("Shape:", annotation_prompts_export.shape)

display(annotation_prompts_export.head())

CSV saved: /content/drive/MyDrive/windows_logs_project/serverC_22_annotation_prompts_30min/serverC_22_annotation_prompts_30min.csv
Shape: (1773, 11)


,AnnotationID,StableID,Split,WindowStart,WindowEnd,SeqLen,Levels,Sources,EventIDs,SuspiciousReason,Prompt
0,0,val_C_2026-04-07 18:00:00_2026-04-07 18:30:00_...,val_C,2026-04-07 18:00:00,2026-04-07 18:30:00,43,"[""Information""]","[""Microsoft-Windows-GroupPolicy"", ""Microsoft-W...","[""1502"", ""158"", ""7036"", ""7040""]",,Ты эксперт по анализу Windows Event Logs и обн...
1,1,val_C_2026-04-07 18:30:00_2026-04-07 19:00:00_...,val_C,2026-04-07 18:30:00,2026-04-07 19:00:00,35,"[""Information""]","[""Service Control Manager""]","[""7036""]",,Ты эксперт по анализу Windows Event Logs и обн...
2,2,val_C_2026-04-07 19:00:00_2026-04-07 19:30:00_...,val_C,2026-04-07 19:00:00,2026-04-07 19:30:00,43,"[""Information""]","[""Service Control Manager""]","[""7036""]",,Ты эксперт по анализу Windows Event Logs и обн...
3,3,val_C_2026-04-07 19:30:00_2026-04-07 20:00:00_...,val_C,2026-04-07 19:30:00,2026-04-07 20:00:00,48,"[""Information""]","[""Service Control Manager""]","[""7036""]",,Ты эксперт по анализу Windows Event Logs и обн...
4,4,val_C_2026-04-07 20:00:00_2026-04-07 20:30:00_...,val_C,2026-04-07 20:00:00,2026-04-07 20:30:00,57,"[""Information""]","[""Microsoft-Windows-GroupPolicy"", ""Microsoft-W...","[""1502"", ""158"", ""7036"", ""7040""]",,Ты эксперт по анализу Windows Event Logs и обн...


In [ ]:
with open(ANNOTATION_PROMPTS_JSONL_PATH, "w", encoding="utf-8") as f:
    for _, row in annotation_prompts.iterrows():
        record = {
            "AnnotationID": int(row["AnnotationID"]),
            "StableID": str(row["StableID"]),
            "Split": str(row["Split"]),
            "WindowStart": str(row["WindowStart"]),
            "WindowEnd": str(row["WindowEnd"]),
            "SeqLen": int(row["SeqLen"]),
            "Prompt": row["Prompt"]
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("JSONL saved:", ANNOTATION_PROMPTS_JSONL_PATH)

JSONL saved: /content/drive/MyDrive/windows_logs_project/serverC_22_annotation_prompts_30min/serverC_22_annotation_prompts_30min.jsonl


In [ ]:
print(annotation_prompts.loc[0, "Prompt"])

Ты эксперт по анализу Windows Event Logs и обнаружению аномалий в последовательностях событий.

Нужно разметить одну последовательность событий из журналов Windows System/Application.

Классы:

0 = normal
Последовательность похожа на обычную активность системы: информационные события, обычный запуск/остановка служб, Windows Update, BITS, Group Policy, WMI/SCM без признаков сбоя.

1 = anomaly
Последовательность содержит признаки сбоя или нештатного поведения: Error/Critical, Windows Error Reporting, ошибки SQL Server, failed login, unexpected shutdown/restart, service terminated unexpectedly, audit log cleared, repeated crash reports, массовая остановка критичных служб или другая явно подозрительная цепочка.

Важно:
- Не считай Service Control Manager 7036 сам по себе аномалией.
- Не считай обычные циклы WMI Performance Adapter running/stopped аномалией без дополнительных признаков сбоя.
- Не считай Windows Update, BITS, Group Policy или AppX активность аномалией, если нет Error/Critica

In [ ]:
ANNOTATION_RESULTS_TEMPLATE_PATH = f"{PROMPT_EXPORT_DIR}/serverC_22_annotation_results_template_binary_30min.csv"

annotation_results_template = annotation_prompts[[
    "AnnotationID",
    "StableID",
    "Split",
    "WindowStart",
    "WindowEnd",
    "SeqLen"
]].copy()

annotation_results_template["label"] = None
annotation_results_template["anomaly_score"] = None
annotation_results_template["confidence"] = None
annotation_results_template["key_events"] = None
annotation_results_template["reason"] = None
annotation_results_template["annotator"] = None

annotation_results_template.to_csv(
    ANNOTATION_RESULTS_TEMPLATE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Binary annotation results template saved:", ANNOTATION_RESULTS_TEMPLATE_PATH)

display(annotation_results_template.head())

Binary annotation results template saved: /content/drive/MyDrive/windows_logs_project/serverC_22_annotation_prompts_30min/serverC_22_annotation_results_template_binary_30min.csv


,AnnotationID,StableID,Split,WindowStart,WindowEnd,SeqLen,label,anomaly_score,confidence,key_events,reason,annotator
0,0,val_C_2026-04-07 18:00:00_2026-04-07 18:30:00_...,val_C,2026-04-07 18:00:00,2026-04-07 18:30:00,43,None,None,None,None,None,None
1,1,val_C_2026-04-07 18:30:00_2026-04-07 19:00:00_...,val_C,2026-04-07 18:30:00,2026-04-07 19:00:00,35,None,None,None,None,None,None
2,2,val_C_2026-04-07 19:00:00_2026-04-07 19:30:00_...,val_C,2026-04-07 19:00:00,2026-04-07 19:30:00,43,None,None,None,None,None,None
3,3,val_C_2026-04-07 19:30:00_2026-04-07 20:00:00_...,val_C,2026-04-07 19:30:00,2026-04-07 20:00:00,48,None,None,None,None,None,None
4,4,val_C_2026-04-07 20:00:00_2026-04-07 20:30:00_...,val_C,2026-04-07 20:00:00,2026-04-07 20:30:00,57,None,None,None,None,None,None


In [ ]:
val_C.to_pickle(f"{EXPORT_DIR}/val_C.pkl")
test_C.to_pickle(f"{EXPORT_DIR}/test_C.pkl")

val_C_raw.to_pickle(f"{EXPORT_DIR}/val_C_raw.pkl")
test_C_raw.to_pickle(f"{EXPORT_DIR}/test_C_raw.pkl")

annotation_df_C.to_pickle(f"{EXPORT_DIR}/annotation_df_C.pkl")

with open(f"{EXPORT_DIR}/template_to_id.pkl", "wb") as f:
    pickle.dump(template_to_id, f)

with open(f"{EXPORT_DIR}/id_to_template.pkl", "wb") as f:
    pickle.dump(id_to_template, f)

with open(f"{EXPORT_DIR}/template_to_id.json", "w", encoding="utf-8") as f:
    json.dump(template_to_id, f, ensure_ascii=False, indent=2)

with open(f"{EXPORT_DIR}/id_to_template.json", "w", encoding="utf-8") as f:
    json.dump(id_to_template, f, ensure_ascii=False, indent=2)

print("Server C pickle and dictionary files saved.")

Server C pickle and dictionary files saved.


In [ ]:
def save_df_csv_json_lists(df: pd.DataFrame, path: str):
    df_to_save = df.copy()

    list_cols = [
        "Features",
        "TimeInterval",
        "LogNames",
        "Sources",
        "Levels",
        "EventIDs",
        "RawMessages",
        "ReviewTemplates",
        "EventLogNames",
        "EventSources",
        "EventLevels",
        "EventIDsPerEvent",
        "EventTokens"
    ]

    for col in list_cols:
        if col in df_to_save.columns:
            df_to_save[col] = df_to_save[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
            )

    df_to_save.to_csv(path, index=False, encoding="utf-8-sig")

In [ ]:
save_df_csv_json_lists(val_C, f"{EXPORT_DIR}/val_C.csv")
save_df_csv_json_lists(test_C, f"{EXPORT_DIR}/test_C.csv")
save_df_csv_json_lists(annotation_df_C, f"{EXPORT_DIR}/annotation_df_C.csv")

print("Server C CSV files saved.")

Server C CSV files saved.


In [ ]:
LOCAL_SERVER_C_DATA_PATH = f"{EXPORT_DIR}/serverC_22_data_30min.pkl"

server_c_data = {
    "params": {
        "VERSION": VERSION,
        "WINDOW_SIZE": WINDOW_SIZE,
        "MIN_SEQ_LEN": MIN_SEQ_LEN,
        "MAX_CHUNK_LEN": MAX_CHUNK_LEN,
        "STRIDE": STRIDE,
        "VAL_C_SIZE": VAL_C_SIZE,
        "MAX_VAL_C_WINDOWS": MAX_VAL_C_WINDOWS,
        "SERVER_NAME": SERVER_NAME,
        "SERVER_C_START_DATE": SERVER_C_START_DATE,
        "SERVER_C_END_DATE": SERVER_C_END_DATE,
    },
    "val_C": val_C,
    "test_C": test_C,
    "annotation_df_C": annotation_df_C,
    "template_to_id": template_to_id,
    "id_to_template": id_to_template
}

with open(LOCAL_SERVER_C_DATA_PATH, "wb") as f:
    pickle.dump(server_c_data, f)

print("Server C data saved:", LOCAL_SERVER_C_DATA_PATH)

Server C data saved: /content/drive/MyDrive/windows_logs_project/serverC_22_export_30min/serverC_22_data_30min.pkl


In [ ]:
with open(LOCAL_SERVER_C_DATA_PATH, "rb") as f:
    check_data = pickle.load(f)

print(check_data.keys())
print("val_C:", check_data["val_C"].shape)
print("test_C:", check_data["test_C"].shape)
print("annotation_df_C:", check_data["annotation_df_C"].shape)

dict_keys(['params', 'val_C', 'test_C', 'annotation_df_C', 'template_to_id', 'id_to_template'])
val_C: (300, 31)
test_C: (1473, 31)
annotation_df_C: (1773, 33)


In [ ]:
THREE_LLM_TEMPLATE_PATH = (
    f"{PROMPT_EXPORT_DIR}/serverC_22_annotation_results_3llm_30min.csv"
)

annotation_results_3llm = annotation_prompts[[
    "AnnotationID",
    "StableID",
    "Split",
    "WindowStart",
    "WindowEnd",
    "SeqLen"
]].copy()

for model_name in ["llm1", "llm2", "llm3"]:
    annotation_results_3llm[f"{model_name}_label"] = None
    annotation_results_3llm[f"{model_name}_score"] = None
    annotation_results_3llm[f"{model_name}_confidence"] = None
    annotation_results_3llm[f"{model_name}_key_events"] = None
    annotation_results_3llm[f"{model_name}_reason"] = None
    annotation_results_3llm[f"{model_name}_annotator"] = None

annotation_results_3llm.to_csv(
    THREE_LLM_TEMPLATE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("3-LLM annotation template saved:", THREE_LLM_TEMPLATE_PATH)
display(annotation_results_3llm.head())

3-LLM annotation template saved: /content/drive/MyDrive/windows_logs_project/serverC_22_annotation_prompts_30min/serverC_22_annotation_results_3llm_30min.csv


,AnnotationID,StableID,Split,WindowStart,WindowEnd,SeqLen,llm1_label,llm1_score,llm1_confidence,llm1_key_events,...,llm2_confidence,llm2_key_events,llm2_reason,llm2_annotator,llm3_label,llm3_score,llm3_confidence,llm3_key_events,llm3_reason,llm3_annotator
0,0,val_C_2026-04-07 18:00:00_2026-04-07 18:30:00_...,val_C,2026-04-07 18:00:00,2026-04-07 18:30:00,43,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,1,val_C_2026-04-07 18:30:00_2026-04-07 19:00:00_...,val_C,2026-04-07 18:30:00,2026-04-07 19:00:00,35,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,2,val_C_2026-04-07 19:00:00_2026-04-07 19:30:00_...,val_C,2026-04-07 19:00:00,2026-04-07 19:30:00,43,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,3,val_C_2026-04-07 19:30:00_2026-04-07 20:00:00_...,val_C,2026-04-07 19:30:00,2026-04-07 20:00:00,48,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,4,val_C_2026-04-07 20:00:00_2026-04-07 20:30:00_...,val_C,2026-04-07 20:00:00,2026-04-07 20:30:00,57,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
